In [1]:
from comet_ml import start, login
from comet_ml.integration.pytorch import log_model, watch

In [ ]:
import kagglehub
import numpy as np
from torch import nn
import matplotlib.pyplot as plt 
import pandas as pd
import os
from pathlib import Path
import torchvision.io as io
from torch.utils.data import DataLoader, Dataset, random_split, Subset, RandomSampler, SubsetRandomSampler
from torchvision import transforms, utils
import torch
import torch.nn.functional as F
from sklearn import svm, model_selection, metrics
import math

In [3]:
DEBUG = False

device = (
    "cpu"
    if DEBUG
    else "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

Using cpu device


In [4]:

dir = "./data"

path = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset", output_dir=dir + "/brain-tumor-mri-dataset")
print("Path to dataset files:", path)

path = kagglehub.dataset_download("tawsifurrahman/covid19-radiography-database", output_dir=dir+"/covid19-radiography-database")
print("Path to dataset files:", path)

Path to dataset files: ./data/brain-tumor-mri-dataset
Path to dataset files: ./data/covid19-radiography-database


In [5]:
class scans_dataset(Dataset):
    def __init__(self, *, dataset, library = "torch"):
        self.dataset = dataset
        self.df = self.build_dataframe(dataset)
        self.length = self.df.shape[0]
        self.library = library

    def __getitem__(self, index):
        item = self.df.iloc[index]
        scan = io.read_image(item['scan'])
        crop = transforms.Resize(size=(256,256))
        if self.dataset == "covid":
            scan = crop(scan)
            scan = scan[0:1, :, :]
        else:
            target = (3,256,256)
            scan = self.__pad__(scan, target)
        return scan, item['class']
    
    def __len__(self):
        return self.df.shape[0]
        
    def __pad__(self, tensor, target_shape):
    # Calculate padding for each dimension in reverse order
    # (starts from the last dimension)
        if self.library == "torch":
            paddings = []
            for cur, target in zip(reversed(tensor.shape), reversed(target_shape)):
                diff = target - cur
                paddings.extend([0, diff])  # [pad_left, pad_right]
        
            return F.pad(tensor, tuple(paddings))
            
    def build_dataframe(self, dataset="covid"):
        df = pd.DataFrame()
        if dataset == "tumors":
            self.channels = 3
            classes = {"glioma":0, "meningioma":1, "notumor": 2, "pituitary":3}
            path = "data/brain-tumor-mri-dataset"
            for t in ["Training", "Testing"]:
                for f in [d for d in os.listdir(path + "/" + t) if os.path.isdir(path + "/" + t + "/" + d)]:
                    files = [f'{path}/{t}/{f}/{d}' for d in os.listdir(path + f'/{t}/{f}')]
                    label = Path(f).name
                    df = pd.concat([df, pd.DataFrame({'testing': t=="Testing", 'label': label, 'class': classes[label], 'scan': files})], ignore_index=True)
        elif dataset == "covid":
            self.channels = 1
            classes = {"COVID": 0, "Lung_Opacity":1, "Normal":2, "Viral Pneumonia":3}
            path = "data/covid19-radiography-database/COVID-19_Radiography_Dataset"
            for f in [d for d in os.listdir(path) if os.path.isdir(path+ "/"+ d)]:
                images = [f'{path}/{f}/images/{d}' for d in os.listdir(path + f'/{f}/images')]
                masks = [f'{path}/{f}/masks/{d}' for d in os.listdir(path+f'/{f}/masks')]
                label = Path(f).name
                df = pd.concat([df, pd.DataFrame({'label': label, 'class': classes[label], 'scan': images, 'mask': masks})], ignore_index=True)
        else:
            raise ValueError("No such dataset")
        
        return df
            

In [61]:
class ResidualLayer(nn.Module):
    def __init__(self, in_channels, out_channels, h_params):
        super(ResidualLayer, self).__init__()

        self.down_scale_flag = in_channels != out_channels
        self.mish = nn.Mish()
        kernel_size = h_params['kernel_size']
        padding = 'same'
        
    

        self.conv = nn.Conv2d(
            in_channels=in_channels,
            out_channels = out_channels,
            kernel_size=(kernel_size,kernel_size),
            dilation= h_params['dilation'],
            padding=padding
        )
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv_2 = nn.Conv2d(
            in_channels=out_channels,
            out_channels = out_channels,
            kernel_size=(kernel_size, kernel_size),
            dilation=h_params['dilation'],
            padding=padding
        )
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.down_scale = nn.Sequential(
            nn.Conv2d(
                in_channels=in_channels,
                out_channels=out_channels,
                kernel_size=(kernel_size, kernel_size),
                dilation=h_params['dilation'],
                padding= padding
            ),
            nn.BatchNorm2d(out_channels)
        )

        
    def forward(self, x):
        identity = self.down_scale(x) if self.down_scale_flag else x
        x = self.conv(x)
        x = self.bn1(x)
        x = self.mish(x)
        x = self.conv_2(x)
        x = self.bn2(x)
        x = self.mish(x + identity)
        return x
        

In [59]:
class scan_cnn(nn.Module):
    def __init__(self, n_classes, input_channels, image_size: tuple, h_params):
        super(scan_cnn, self).__init__()
        self.layers = 4
        
        self.conv_block = nn.Conv2d(in_channels=input_channels,
                                    out_channels=16,
                                    kernel_size=(h_params['kernel_size'], h_params['kernel_size']),
                                    padding='same')
        self.mish = nn.Mish()
        self.norm = nn.BatchNorm2d(16)
        self.dropout = nn.Dropout(p = h_params['dropout'])
        self.residual_block_1 = nn.Sequential(
            ResidualLayer(16,16, h_params),
            ResidualLayer(16,16, h_params),
            ResidualLayer(16,16, h_params),
            nn.AvgPool2d((h_params['kernel_size'], h_params['kernel_size']), 
                         stride = (h_params['kernel_size'], h_params['kernel_size']))
        )
        self.residual_block_2 = nn.Sequential(
            ResidualLayer(16, 32, h_params),
            ResidualLayer(32, 32, h_params),
            ResidualLayer(32, 32, h_params),
            nn.AvgPool2d((h_params['kernel_size'], h_params['kernel_size']), 
                         stride = (h_params['kernel_size'], h_params['kernel_size']))
        )
        self.residual_block_3 = nn.Sequential(
            ResidualLayer(32, 64, h_params),
            ResidualLayer(64, 64, h_params),
            ResidualLayer(64, 64, h_params),
            nn.AvgPool2d((h_params['kernel_size'], h_params['kernel_size']), 
                         stride = (h_params['kernel_size'], h_params['kernel_size']))
        )
        self.residual_block_4 = nn.Sequential(
            ResidualLayer(64, 128, h_params),
            ResidualLayer(128, 128, h_params),
            ResidualLayer(128, 128, h_params),
            nn.AvgPool2d((h_params['kernel_size'], h_params['kernel_size']), 
                         stride= (h_params['kernel_size'], h_params['kernel_size']))
        )

        self.flatten = nn.Flatten(1)
        in_features = image_size[0] // h_params['kernel_size']**4 * image_size[1] // h_params['kernel_size']**4 * 128
        
        self.linear = nn.Linear(
            in_features = in_features,
            out_features = n_classes
        )
    
    def forward(self, x):
        x = self.conv_block(x)
        x = self.norm(x)
        x = self.mish(x)

        x = self.residual_block_1(x)
        x = self.residual_block_2(x)
        x = self.residual_block_3(x)
        x = self.residual_block_4(x)
        
        x = self.dropout(x)

        x = self.flatten(x)
        x = self.linear(x)

        return x

In [ ]:
def train_cnn(dataset, image_model, h_params, experiment):
    sum_loss = 0
    losses = []
    i = 0
    optimizer = torch.optim.Adam(image_model.parameters(), lr=h_params['learning_rate'])
    loss_fn = torch.nn.NLLLoss()
    log_softmax = nn.LogSoftmax(dim=0)
    

    for epoch in range(h_params['epochs']):
        total = 0
        correct = 0
        experiment.log_current_epoch(epoch)
        for (X, y) in dataset:
            X = X.float()
            X, Y = X.to(device), y.to(device)
            optimizer.zero_grad()

            output = image_model(X)
            _, predicted = torch.max(output.data, 1)
            loss = loss_fn(predicted, Y.long())
  
            loss.backward()
            optimizer.step()
            losses.append(loss.item())

            total += Y.size(0)
            correct += float((predicted == Y.data).sum())

            # Log accuracy to Comet.ml
            experiment.log_metric("train accuracy", 100 * correct / total, step=i)
            experiment.log_metric("train loss", loss, step=i)

            
            i += 1
            
    return losses
    

In [67]:
def test_cnn(dataset, model, experiment):
    model.eval()
    with torch.no_grad():
        for X,y in dataset:
            X, y = X.to(device), y.to(device)
            
            output = model(X)
            _, predicted = torch.max(output.data, 1)
            
            
            total += y.size(0)
            correct += float((predicted == y.data).sum())
            
            experiment.log("test accuracy", 100 * correct/total)
            
            
            

In [58]:
def train_svm(dataset):
    clf = svm.SVC()
    X, y = [], []
    for i in range(0, dataset.length, dataset.length//600):
        img, label = dataset.__getitem__(i)
        img = img.flatten().tolist()
        X.append(img)
        y.append(label.item(0))
    X_train, X_test, y_train, y_test = model_selection.train_test_split(X, y, train_size=.8)
    clf.fit(X_train, y_train)
    print(dataset.length)
    return clf, X_test, y_test

In [66]:
tumors_data = scans_dataset(dataset="tumors", library="torch")
covid_data = scans_dataset(dataset="covid", library="torch")
generator = torch.Generator()

In [11]:
login()

In [68]:
kernel_sizes = [3,5,7]
dilation = [1, 2, 3]
epochs = [1000, 10000, 100000],
learning_rates = [.001, .01, .1]
batch_sizes = [20]
dropouts = [.05, .1, .2]
datasets = [tumors_data, covid_data]
h_params = {'dataset':'', 'input_size':(256,256), 'learning_rate': .2, 'epochs': 1, 'batch_size': 0, 'kernel_size': 0, 'dilation': 1, 'dropout': 0}
for b in batch_sizes:
    for k in kernel_sizes:
        for dl in dilation:
            for ds in dropouts:
                for data in datasets:
                    experiment = start(project="ml-project")
                    h_params['batch_size'], h_params['kernel_size'], h_params['dilation'], h_params['dropout'] = b, k, dl, ds
                    h_params['dataset'] = data.dataset            
                    experiment.log_parameters(h_params)
                    
                    train_indices, val_indices = model_selection.train_test_split(
                        list(range(len(data))),
                        test_size=.2,
                        stratify=data.df['class'],
                        random_state=42
                    )
                    
                    train_set = Subset(data, train_indices)
                    val_set = Subset(data, val_indices)
                    
                    train_dataloader = DataLoader(data, batch_size=b, sampler=RandomSampler(train_set, generator=generator, num_samples=2000))
                    val_dataloader = DataLoader(data, batch_size=b, sampler=RandomSampler(val_set, generator=generator, num_samples=300))
                    
                    image_model = scan_cnn(n_classes=4, input_channels=data.channels, image_size=(256,256), h_params=h_params).to(device)
                    losses = train_cnn(train_dataloader, image_model, h_params, experiment)
                    
                    test_cnn(val_dataloader, image_model)
                    
                    experiment.end()

COMET INFO: An experiment with the same configuration options is already running and will be reused.


KeyboardInterrupt: 

In [ ]:
clf, X_test, y_test = train_svm(dataset)

In [ ]:
print(metrics.classification_report(y_test, clf.predict(X_test)))